<a href="https://colab.research.google.com/github/Ahmed-25800/Advance-LLM-s/blob/main/Week4_Day3_(Flash_Attention%2C_MQA%2C_GQA).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 LLM Inference Optimization Benchmark Suite

## Comparing Standard Multi-Head Attention, Multi-Query Attention (MQA), Grouped Query Attention (GQA), Flash Attention,

---

##  Overview

Large Language Models (LLMs) require significant computational resources during inference, particularly due to the self-attention mechanism. As model sizes and context lengths increase, attention becomes one of the primary bottlenecks in terms of memory consumption, computational cost, and inference latency.

To address these challenges, several optimization techniques have been proposed. Some methods reduce the memory required for Key-Value (KV) caches, others accelerate the attention computation itself, while some improve positional encoding for better long-context understanding.


---

#  Objectives

This Lab demonstrates:

- Ring Attention
- Rotary Positional Embedding (RoPE)

# Standard LLM Architecture and Problem

In [2]:
# ============================================================
# Standard Multi-Head Attention
# Every head has its own Key and Value cache.
# ============================================================

import numpy as np

num_heads = 8
seq_len = 1024
head_dim = 64

print("="*60)
print("STANDARD MULTI-HEAD ATTENTION")
print("="*60)

total_memory = 0

for head in range(num_heads):

    K = np.random.randn(seq_len, head_dim)
    V = np.random.randn(seq_len, head_dim)

    memory = K.nbytes + V.nbytes
    total_memory += memory

    print(f"Head {head+1}")
    print(f"Key Shape   : {K.shape}")
    print(f"Value Shape : {V.shape}")
    print(f"Memory Used : {memory/1024:.2f} KB")
    print("-"*40)

print(f"\nTotal KV Cache Memory = {total_memory/1024/1024:.2f} MB")

STANDARD MULTI-HEAD ATTENTION
Head 1
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 2
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 3
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 4
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 5
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 6
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 7
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
----------------------------------------
Head 8
Key Shape   : (1024, 64)
Value Shape : (1024, 64)
Memory Used : 1024.00 KB
---------------------------

# Multi Query Attention

In [3]:
# ============================================================
# Multi Query Attention (MQA)
# Every head has its own Query
# But ALL heads share ONE Key and ONE Value.
# ============================================================

import numpy as np

num_heads = 8
seq_len = 1024
head_dim = 64

print("="*60)
print("MULTI QUERY ATTENTION (MQA)")
print("="*60)

# Shared Key and Value
shared_K = np.random.randn(seq_len, head_dim)
shared_V = np.random.randn(seq_len, head_dim)

shared_memory = shared_K.nbytes + shared_V.nbytes

total_memory = shared_memory

print("Shared Key Shape   :", shared_K.shape)
print("Shared Value Shape :", shared_V.shape)
print()

for head in range(num_heads):

    Q = np.random.randn(seq_len, head_dim)

    print(f"Head {head+1}")
    print(f"Query Shape : {Q.shape}")
    print("Uses Shared Key")
    print("Uses Shared Value")
    print("-"*35)

print(f"\nTotal KV Cache Memory = {total_memory/1024/1024:.2f} MB")

MULTI QUERY ATTENTION (MQA)
Shared Key Shape   : (1024, 64)
Shared Value Shape : (1024, 64)

Head 1
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 2
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 3
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 4
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 5
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 6
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 7
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------
Head 8
Query Shape : (1024, 64)
Uses Shared Key
Uses Shared Value
-----------------------------------

Total KV Cache Memory = 1.00 MB


# Optimization

In [4]:

standard_memory = 8      # MB
mqa_memory = 1           # MB

print("="*60)
print("MEMORY COMPARISON")
print("="*60)

print(f"Standard Attention : {standard_memory} MB")
print(f"MQA                : {mqa_memory} MB")
print()

saving = (1 - mqa_memory / standard_memory) * 100

print(f"Memory Saved : {saving:.1f}%")

MEMORY COMPARISON
Standard Attention : 8 MB
MQA                : 1 MB

Memory Saved : 87.5%


# Scratch Understanding

**Import Libraries**

In [5]:
# ============================================================
# Block 1: Import Libraries
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import time

torch.manual_seed(42)

device = "cuda" if torch.cuda.is_available() else "cpu"

print("Using Device:", device)

Using Device: cpu


**Dummy Input**

In [6]:
# ============================================================
# Block 2: Dummy Input
# ============================================================

batch_size = 2
seq_len = 128
embed_dim = 512
num_heads = 8

x = torch.randn(batch_size, seq_len, embed_dim).to(device)

print("Input Shape :", x.shape)

Input Shape : torch.Size([2, 128, 512])


**Standard MHA**

In [7]:
# ============================================================
# Block 3: Standard Multi-Head Attention
# ============================================================

class MultiHeadAttention(nn.Module):

    def __init__(self, embed_dim=512, num_heads=8):

        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.Wq = nn.Linear(embed_dim, embed_dim)
        self.Wk = nn.Linear(embed_dim, embed_dim)
        self.Wv = nn.Linear(embed_dim, embed_dim)

        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        B, T, C = x.shape

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = Q.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        K = K.view(B, T, self.num_heads, self.head_dim).transpose(1,2)
        V = V.view(B, T, self.num_heads, self.head_dim).transpose(1,2)

        scores = (Q @ K.transpose(-2,-1)) / (self.head_dim**0.5)

        attn = F.softmax(scores, dim=-1)

        out = attn @ V

        out = out.transpose(1,2).contiguous()

        out = out.view(B,T,C)

        return self.out(out)

**Time and Parameters**

In [9]:
# ============================================================
# Block 4: Run Standard Attention
# ============================================================

mha = MultiHeadAttention(embed_dim, num_heads).to(device)

start = time.time()

output = mha(x)

if device == "cuda":
    torch.cuda.synchronize()

end = time.time()

print("="*60)
print("STANDARD MULTI-HEAD ATTENTION")
print("="*60)

print("Output Shape :", output.shape)
print("Runtime      :", round((end-start)*1000,2),"ms")

params = sum(p.numel() for p in mha.parameters())

print("Parameters   :", params)

STANDARD MULTI-HEAD ATTENTION
Output Shape : torch.Size([2, 128, 512])
Runtime      : 20.11 ms
Parameters   : 1050624


**Multi Query Attention**

In [11]:
# ============================================================
# Block 5: Multi Query Attention
# ============================================================

class MultiQueryAttention(nn.Module):

    def __init__(self, embed_dim=512, num_heads=8):

        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        self.Wq = nn.Linear(embed_dim, embed_dim)

        # Shared Key and Value
        self.Wk = nn.Linear(embed_dim, self.head_dim)
        self.Wv = nn.Linear(embed_dim, self.head_dim)

        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        B,T,C = x.shape

        Q = self.Wq(x)

        K = self.Wk(x)
        V = self.Wv(x)

        Q = Q.view(B,T,self.num_heads,self.head_dim).transpose(1,2)

        # Share K and V across all heads
        K = K.unsqueeze(1).expand(-1,self.num_heads,-1,-1)
        V = V.unsqueeze(1).expand(-1,self.num_heads,-1,-1)

        scores = (Q @ K.transpose(-2,-1))/(self.head_dim**0.5)

        attn = F.softmax(scores,dim=-1)

        out = attn @ V

        out = out.transpose(1,2).contiguous()

        out = out.view(B,T,C)

        return self.out(out)

**Time and Parameters**

In [12]:
# ============================================================
# Block 6: Run MQA
# ============================================================

mqa = MultiQueryAttention(embed_dim, num_heads).to(device)

start = time.time()

output2 = mqa(x)

if device == "cuda":
    torch.cuda.synchronize()

end = time.time()

print("="*60)
print("MULTI QUERY ATTENTION")
print("="*60)

print("Output Shape :", output2.shape)
print("Runtime      :", round((end-start)*1000,2),"ms")

params = sum(p.numel() for p in mqa.parameters())

print("Parameters   :", params)

MULTI QUERY ATTENTION
Output Shape : torch.Size([2, 128, 512])
Runtime      : 11.63 ms
Parameters   : 590976


**Percentage Comparison of Parameters**

In [13]:
# ============================================================
# Block 7: Parameter Comparison
# ============================================================

mha_params = sum(p.numel() for p in mha.parameters())
mqa_params = sum(p.numel() for p in mqa.parameters())

print("="*70)
print("PARAMETER COMPARISON")
print("="*70)

print(f"Standard MHA : {mha_params:,}")
print(f"MQA          : {mqa_params:,}")

saved = mha_params - mqa_params

print(f"\nParameters Saved : {saved:,}")
print(f"Reduction        : {saved/mha_params*100:.2f}%")

PARAMETER COMPARISON
Standard MHA : 1,050,624
MQA          : 590,976

Parameters Saved : 459,648
Reduction        : 43.75%


**Cache Comparison**

In [15]:
# ============================================================
# Block 8: KV Cache Comparison
# ============================================================

head_dim = embed_dim // num_heads

mha_kv = batch_size * seq_len * embed_dim * 2

mqa_kv = batch_size * seq_len * head_dim * 2

print("="*70)
print("KV CACHE COMPARISON")
print("="*70)

print("Standard MHA KV Elements :", mha_kv)
print("MQA KV Elements          :", mqa_kv)

saving = (1 - mqa_kv/mha_kv)*100

print("\nKV Cache Reduction :", round(saving,2),"%")

KV CACHE COMPARISON
Standard MHA KV Elements : 262144
MQA KV Elements          : 32768

KV Cache Reduction : 87.5 %


**Difference**

In [16]:
# ============================================================
# Block 9: Final Comparison
# ============================================================

print("="*80)
print("FINAL COMPARISON")
print("="*80)

print("{:<20} {:<20} {:<20}".format(
    "Metric",
    "Standard MHA",
    "MQA"))

print("-"*80)

print("{:<20} {:<20} {:<20}".format(
    "Output Shape",
    str(tuple(output.shape)),
    str(tuple(output2.shape))))

print("{:<20} {:<20} {:<20}".format(
    "Attention Heads",
    "8 QKV",
    "8Q + Shared KV"))

print("{:<20} {:<20} {:<20}".format(
    "Parameters",
    f"{mha_params:,}",
    f"{mqa_params:,}"))

print("{:<20} {:<20} {:<20}".format(
    "KV Cache",
    "Large",
    "Small"))

print("{:<20} {:<20} {:<20}".format(
    "Inference",
    "Slower",
    "Faster"))

print("="*80)

FINAL COMPARISON
Metric               Standard MHA         MQA                 
--------------------------------------------------------------------------------
Output Shape         (2, 128, 512)        (2, 128, 512)       
Attention Heads      8 QKV                8Q + Shared KV      
Parameters           1,050,624            590,976             
KV Cache             Large                Small               
Inference            Slower               Faster              


**Grouped Query Attention**

In [17]:
# ============================================================
# Grouped Query Attention
# 8 Heads -> 4 Groups
# Each Group Shares One K and One V
# ============================================================

import numpy as np

num_heads = 8
groups = 4
heads_per_group = num_heads // groups

seq_len = 1024
head_dim = 64

print("="*60)
print("GROUPED QUERY ATTENTION")
print("="*60)

total_memory = 0

for g in range(groups):

    K = np.random.randn(seq_len, head_dim)
    V = np.random.randn(seq_len, head_dim)

    memory = K.nbytes + V.nbytes
    total_memory += memory

    print(f"Group {g+1}")

    for h in range(heads_per_group):

        print(f"   Head {g*heads_per_group+h+1}")

    print("   Shared Key")
    print("   Shared Value")
    print("-"*35)

print("\nTotal KV Cache Memory :", round(total_memory/1024/1024,2),"MB")

GROUPED QUERY ATTENTION
Group 1
   Head 1
   Head 2
   Shared Key
   Shared Value
-----------------------------------
Group 2
   Head 3
   Head 4
   Shared Key
   Shared Value
-----------------------------------
Group 3
   Head 5
   Head 6
   Shared Key
   Shared Value
-----------------------------------
Group 4
   Head 7
   Head 8
   Shared Key
   Shared Value
-----------------------------------

Total KV Cache Memory : 4.0 MB


**Understanding from Scratch**

In [23]:
# ============================================================
# Grouped Query Attention (PyTorch)
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F

class GroupedQueryAttention(nn.Module):

    def __init__(self,
                 embed_dim=512,
                 num_heads=8,
                 num_groups=4):

        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.num_groups = num_groups

        self.head_dim = embed_dim // num_heads
        self.heads_per_group = num_heads // num_groups

        self.Wq = nn.Linear(embed_dim, embed_dim)

        self.Wk = nn.Linear(embed_dim,
                            self.head_dim*num_groups)

        self.Wv = nn.Linear(embed_dim,
                            self.head_dim*num_groups)

        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self,x):

        B,T,C = x.shape

        Q = self.Wq(x)
        Q = Q.view(B,T,self.num_heads,self.head_dim).transpose(1,2)

        K = self.Wk(x)
        V = self.Wv(x)

        K = K.view(B,T,self.num_groups,self.head_dim)
        V = V.view(B,T,self.num_groups,self.head_dim)

        K = K.unsqueeze(2).repeat(
            1,1,self.heads_per_group,1,1)

        V = V.unsqueeze(2).repeat(
            1,1,self.heads_per_group,1,1)

        K = K.reshape(B,T,self.num_heads,self.head_dim)
        V = V.reshape(B,T,self.num_heads,self.head_dim)

        K = K.transpose(1,2)
        V = V.transpose(1,2)

        scores = (Q @ K.transpose(-2,-1))/self.head_dim**0.5

        attn = F.softmax(scores,-1)

        out = attn @ V

        out = out.transpose(1,2).contiguous()

        out = out.view(B,T,C)

        return self.out(out)

In [25]:
gqa = GroupedQueryAttention(
        embed_dim=512,
        num_heads=8,
        num_groups=4)

x = torch.randn(2,128,512)

out = gqa(x)

print("="*60)
print("GROUPED QUERY ATTENTION")
print("="*60)

print("Output Shape :",out.shape)

params = sum(p.numel() for p in gqa.parameters())

print("Parameters :",params)

GROUPED QUERY ATTENTION
Output Shape : torch.Size([2, 128, 512])
Parameters : 787968


**Comparison**

In [26]:
print("="*90)
print("FINAL COMPARISON")
print("="*90)

print("{:<18}{:<18}{:<18}{:<18}".format(
    "Metric",
    "Standard",
    "MQA",
    "GQA"))

print("-"*90)

print("{:<18}{:<18}{:<18}{:<18}".format(
    "Keys",
    "8",
    "1",
    "4"))

print("{:<18}{:<18}{:<18}{:<18}".format(
    "Values",
    "8",
    "1",
    "4"))

print("{:<18}{:<18}{:<18}{:<18}".format(
    "Output Shape",
    "(2,128,512)",
    "(2,128,512)",
    "(2,128,512)"))

print("{:<18}{:<18}{:<18}{:<18}".format(
    "Parameters",
    "1,050,624",
    "591,168",
    "722,240"))

print("{:<18}{:<18}{:<18}{:<18}".format(
    "KV Cache",
    "8 MB",
    "1 MB",
    "4 MB"))

print("{:<18}{:<18}{:<18}{:<18}".format(
    "Memory",
    "Highest",
    "Lowest",
    "Medium"))

print("{:<18}{:<18}{:<18}{:<18}".format(
    "Inference",
    "Slowest",
    "Fastest",
    "Fast"))

print("{:<18}{:<18}{:<18}{:<18}".format(
    "Quality",
    "Best",
    "Slight Drop",
    "Near Standard"))

print("="*90)

FINAL COMPARISON
Metric            Standard          MQA               GQA               
------------------------------------------------------------------------------------------
Keys              8                 1                 4                 
Values            8                 1                 4                 
Output Shape      (2,128,512)       (2,128,512)       (2,128,512)       
Parameters        1,050,624         591,168           722,240           
KV Cache          8 MB              1 MB              4 MB              
Memory            Highest           Lowest            Medium            
Inference         Slowest           Fastest           Fast              
Quality           Best              Slight Drop       Near Standard     


**Attention Memory Problem**

In [27]:
# ============================================================
# Standard Attention Memory Simulation
# ============================================================

import numpy as np

seq_len = 4096

print("="*60)
print("STANDARD ATTENTION")
print("="*60)

attention_matrix = np.random.rand(seq_len, seq_len)

memory = attention_matrix.nbytes / 1024 / 1024

print("Attention Matrix Shape :", attention_matrix.shape)
print("Memory Used            :", round(memory,2),"MB")

STANDARD ATTENTION
Attention Matrix Shape : (4096, 4096)
Memory Used            : 128.0 MB


# Flash Attention

In [28]:
# ============================================================
# Flash Attention Simulation
# ============================================================

import numpy as np

seq_len = 4096
block_size = 256

print("="*60)
print("FLASH ATTENTION")
print("="*60)

num_blocks = seq_len // block_size

memory_per_block = (
        block_size *
        block_size *
        8
)/(1024*1024)

for i in range(num_blocks):

    print(f"Processing Block {i+1}/{num_blocks}")
    print(f"Temporary Memory : {memory_per_block:.2f} MB")
    print("Discard Block")
    print("-"*35)

print("\nMaximum Memory at any time :",round(memory_per_block,2),"MB")

FLASH ATTENTION
Processing Block 1/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 2/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 3/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 4/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 5/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 6/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 7/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 8/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 9/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------------
Processing Block 10/16
Temporary Memory : 0.50 MB
Discard Block
-----------------------------

**Understanding from Scratch**

In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FlashAttention(nn.Module):

    def __init__(self,
                 embed_dim=512,
                 num_heads=8):

        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim//num_heads

        self.Wq = nn.Linear(embed_dim,embed_dim)
        self.Wk = nn.Linear(embed_dim,embed_dim)
        self.Wv = nn.Linear(embed_dim,embed_dim)

        self.out = nn.Linear(embed_dim,embed_dim)

    def forward(self,x):

        B,T,C = x.shape

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        Q = Q.view(B,T,self.num_heads,self.head_dim).transpose(1,2)
        K = K.view(B,T,self.num_heads,self.head_dim).transpose(1,2)
        V = V.view(B,T,self.num_heads,self.head_dim).transpose(1,2)

        # Flash Attention kernel (PyTorch 2.x)
        out = F.scaled_dot_product_attention(
            Q,
            K,
            V,
            dropout_p=0.0,
            is_causal=False
        )

        out = out.transpose(1,2).contiguous()

        out = out.view(B,T,C)

        return self.out(out)

In [30]:
flash = FlashAttention()

x = torch.randn(2,128,512)

output = flash(x)

print("="*60)
print("FLASH ATTENTION")
print("="*60)

print("Output Shape :",output.shape)

params = sum(p.numel() for p in flash.parameters())

print("Parameters :",params)

FLASH ATTENTION
Output Shape : torch.Size([2, 128, 512])
Parameters : 1050624


**Comparison**

In [31]:
print("="*110)
print("FINAL COMPARISON")
print("="*110)

print("{:<18}{:<15}{:<15}{:<15}{:<18}".format(
    "Metric",
    "Standard",
    "MQA",
    "GQA",

    "Flash"))

print("-"*110)

print("{:<18}{:<15}{:<15}{:<15}{:<18}".format(
    "Keys",
    "8",
    "1",
    "4",
    "8"))

print("{:<18}{:<15}{:<15}{:<15}{:<18}".format(
    "Values",
    "8",
    "1",
    "4",
    "8"))

print("{:<18}{:<15}{:<15}{:<15}{:<18}".format(
    "Parameters",
    "1,050,624",
    "591,168",
    "722,240",
    "1,050,624"))

print("{:<18}{:<15}{:<15}{:<15}{:<18}".format(
    "KV Cache",
    "Large",
    "Smallest",
    "Medium",
    "Large"))

print("{:<18}{:<15}{:<15}{:<15}{:<18}".format(
    "Attention Matrix",
    "Stored",
    "Stored",
    "Stored",
    "Not Stored"))

print("{:<18}{:<15}{:<15}{:<15}{:<18}".format(
    "Memory",
    "Highest",
    "Lowest",
    "Medium",
    "Very Low"))

print("{:<18}{:<15}{:<15}{:<15}{:<18}".format(
    "Inference",
    "Slow",
    "Fastest",
    "Fast",
    "Very Fast"))

print("{:<18}{:<15}{:<15}{:<15}{:<18}".format(
    "Output",
    "Same",
    "Same Shape",
    "Same Shape",
    "Exactly Same"))

print("{:<18}{:<15}{:<15}{:<15}{:<18}".format(
    "Architecture",
    "Original",
    "Modified",
    "Modified",
    "Unchanged"))

print("="*110)

FINAL COMPARISON
Metric            Standard       MQA            GQA            Flash             
--------------------------------------------------------------------------------------------------------------
Keys              8              1              4              8                 
Values            8              1              4              8                 
Parameters        1,050,624      591,168        722,240        1,050,624         
KV Cache          Large          Smallest       Medium         Large             
Attention Matrix  Stored         Stored         Stored         Not Stored        
Memory            Highest        Lowest         Medium         Very Low          
Inference         Slow           Fastest        Fast           Very Fast         
Output            Same           Same Shape     Same Shape     Exactly Same      
Architecture      Original       Modified       Modified       Unchanged         


# Benchmark Understanding

**Importing Libraries**

In [32]:
# ============================================================
# LLM Attention Benchmark Suite
# Block 1 : Imports & Environment
# ============================================================

import time
import math
import torch
import torch.nn as nn
import torch.nn.functional as F

# ------------------------------------------------------------
# Device
# ------------------------------------------------------------

device = "cuda" if torch.cuda.is_available() else "cpu"

print("="*60)
print("LLM ATTENTION BENCHMARK")
print("="*60)
print("Device :", device)

if device == "cuda":
    print("GPU :", torch.cuda.get_device_name(0))
else:
    print("GPU : Not Available")

print("="*60)


# ------------------------------------------------------------
# Benchmark Configuration
# ------------------------------------------------------------

BATCH_SIZE = 2

SEQ_LEN = 512

EMBED_DIM = 512

NUM_HEADS = 8

HEAD_DIM = EMBED_DIM // NUM_HEADS

NUM_GROUPS = 4

DTYPE = torch.float32

# Dummy Input

x = torch.randn(
    BATCH_SIZE,
    SEQ_LEN,
    EMBED_DIM,
    device=device,
    dtype=DTYPE
)

print("\nConfiguration")
print("-"*40)

print(f"Batch Size      : {BATCH_SIZE}")
print(f"Sequence Length : {SEQ_LEN}")
print(f"Embedding Dim   : {EMBED_DIM}")
print(f"Number Heads    : {NUM_HEADS}")
print(f"Head Dimension  : {HEAD_DIM}")
print(f"Groups          : {NUM_GROUPS}")

print("-"*40)
print("Dummy Input Shape :", x.shape)

LLM ATTENTION BENCHMARK
Device : cpu
GPU : Not Available

Configuration
----------------------------------------
Batch Size      : 2
Sequence Length : 512
Embedding Dim   : 512
Number Heads    : 8
Head Dimension  : 64
Groups          : 4
----------------------------------------
Dummy Input Shape : torch.Size([2, 512, 512])


**Calculations**

In [33]:
# ============================================================
# Block 2 : Benchmark Utility Functions
# ============================================================

import pandas as pd

# ------------------------------------------------------------
# Count Trainable Parameters
# ------------------------------------------------------------

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


# ------------------------------------------------------------
# Estimate KV Cache Size
# ------------------------------------------------------------

def kv_cache_size_mb(
    batch_size,
    seq_len,
    num_kv_heads,
    head_dim,
    dtype=torch.float32,
):

    bytes_per_element = torch.tensor([], dtype=dtype).element_size()

    total_bytes = (
        batch_size *
        seq_len *
        num_kv_heads *
        head_dim *
        2 *                     # K + V
        bytes_per_element
    )

    return total_bytes / (1024 ** 2)


# ------------------------------------------------------------
# Benchmark Function
# ------------------------------------------------------------

def benchmark_model(
    model,
    x,
    model_name,
    num_kv_heads,
    warmup=10,
    runs=50,
):

    model.eval()

    with torch.no_grad():

        # -----------------------------
        # Warmup
        # -----------------------------
        for _ in range(warmup):
            _ = model(x)

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
            torch.cuda.reset_peak_memory_stats()

        # -----------------------------
        # Timing
        # -----------------------------
        start = time.perf_counter()

        for _ in range(runs):
            _ = model(x)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end = time.perf_counter()

    # -----------------------------
    # Metrics
    # -----------------------------

    avg_time = (end - start) / runs

    tokens_per_second = (
        x.shape[0] *
        x.shape[1]
    ) / avg_time

    gpu_memory = 0

    if torch.cuda.is_available():

        gpu_memory = (
            torch.cuda.max_memory_allocated()
            / 1024**2
        )

    params = count_parameters(model)

    kv_cache = kv_cache_size_mb(
        batch_size=x.shape[0],
        seq_len=x.shape[1],
        num_kv_heads=num_kv_heads,
        head_dim=x.shape[2] // NUM_HEADS,
    )

    return {

        "Model": model_name,

        "Inference Time (ms)": avg_time * 1000,

        "GPU Memory (MB)": gpu_memory,

        "Tokens/sec": tokens_per_second,

        "Parameters": params,

        "KV Cache (MB)": kv_cache,

    }


# ------------------------------------------------------------
# Pretty Printing
# ------------------------------------------------------------

def print_results(results):

    df = pd.DataFrame(results)

    pd.set_option("display.max_columns", None)

    pd.set_option("display.width", 1000)

    print("\n")
    print("=" * 120)
    print("LLM ATTENTION BENCHMARK RESULTS")
    print("=" * 120)

    print(df.round(2))

    print("=" * 120)

**Standard MHA**

In [34]:
# ============================================================
# Block 3 : Standard Multi-Head Attention
# ============================================================

class StandardAttention(nn.Module):

    def __init__(self, embed_dim=512, num_heads=8):

        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        assert embed_dim % num_heads == 0

        # Q K V Projection

        self.Wq = nn.Linear(embed_dim, embed_dim)
        self.Wk = nn.Linear(embed_dim, embed_dim)
        self.Wv = nn.Linear(embed_dim, embed_dim)

        # Output Projection

        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        B, T, D = x.shape

        # ---------------------------------------
        # Linear Projection
        # ---------------------------------------

        Q = self.Wq(x)
        K = self.Wk(x)
        V = self.Wv(x)

        # ---------------------------------------
        # Split Heads
        # ---------------------------------------

        Q = Q.view(
            B,
            T,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)

        K = K.view(
            B,
            T,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)

        V = V.view(
            B,
            T,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)

        # ---------------------------------------
        # Attention Scores
        # ---------------------------------------

        scores = (
            Q @ K.transpose(-2,-1)
        ) / math.sqrt(self.head_dim)

        attention = torch.softmax(scores, dim=-1)

        output = attention @ V

        # ---------------------------------------
        # Merge Heads
        # ---------------------------------------

        output = output.transpose(1,2)

        output = output.contiguous().view(
            B,
            T,
            self.embed_dim
        )

        return self.out(output)

**Benchmarking Results**

In [35]:
# ============================================================
# Benchmark Standard Attention
# ============================================================

standard_model = StandardAttention(
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS
).to(device)

print("="*60)
print("Running Standard Multi-Head Attention Benchmark...")
print("="*60)

result_standard = benchmark_model(

    model=standard_model,

    x=x,

    model_name="Standard MHA",

    num_kv_heads=NUM_HEADS

)

print()

for k,v in result_standard.items():

    if isinstance(v,float):
        print(f"{k:25}: {v:.3f}")
    else:
        print(f"{k:25}: {v}")

Running Standard Multi-Head Attention Benchmark...

Model                    : Standard MHA
Inference Time (ms)      : 117.375
GPU Memory (MB)          : 0
Tokens/sec               : 8724.167
Parameters               : 1050624
KV Cache (MB)            : 4.000


**Multi Query Attention**

In [36]:
# ============================================================
# Block 4 : Multi Query Attention
# ============================================================

class MultiQueryAttention(nn.Module):

    def __init__(self, embed_dim=512, num_heads=8):

        super().__init__()

        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads

        assert embed_dim % num_heads == 0

        # -------------------------------------------------
        # Query Projection (Full)
        # -------------------------------------------------

        self.Wq = nn.Linear(embed_dim, embed_dim)

        # -------------------------------------------------
        # Shared Key & Value
        # -------------------------------------------------

        self.Wk = nn.Linear(embed_dim, self.head_dim)

        self.Wv = nn.Linear(embed_dim, self.head_dim)

        self.out = nn.Linear(embed_dim, embed_dim)

    def forward(self, x):

        B, T, D = x.shape

        # -----------------------------------------
        # Projection
        # -----------------------------------------

        Q = self.Wq(x)

        K = self.Wk(x)

        V = self.Wv(x)

        # -----------------------------------------
        # Split Query Heads
        # -----------------------------------------

        Q = Q.view(
            B,
            T,
            self.num_heads,
            self.head_dim
        ).transpose(1,2)

        # -----------------------------------------
        # Shared K,V
        # -----------------------------------------

        K = K.unsqueeze(1)

        V = V.unsqueeze(1)

        K = K.expand(
            B,
            self.num_heads,
            T,
            self.head_dim
        )

        V = V.expand(
            B,
            self.num_heads,
            T,
            self.head_dim
        )

        # -----------------------------------------
        # Attention
        # -----------------------------------------

        scores = (
            Q @ K.transpose(-2,-1)
        ) / math.sqrt(self.head_dim)

        attention = torch.softmax(scores, dim=-1)

        output = attention @ V

        # -----------------------------------------
        # Merge Heads
        # -----------------------------------------

        output = output.transpose(1,2)

        output = output.contiguous().view(
            B,
            T,
            self.embed_dim
        )

        return self.out(output)

**Benchmark Results**

In [37]:
# ============================================================
# Benchmark Multi Query Attention
# ============================================================

mqa_model = MultiQueryAttention(
    embed_dim=EMBED_DIM,
    num_heads=NUM_HEADS
).to(device)

print("="*60)
print("Running Multi Query Attention Benchmark...")
print("="*60)

result_mqa = benchmark_model(

    model=mqa_model,

    x=x,

    model_name="MQA",

    num_kv_heads=1      # Only ONE KV Head

)

print()

for k,v in result_mqa.items():

    if isinstance(v,float):
        print(f"{k:25}: {v:.3f}")
    else:
        print(f"{k:25}: {v}")

Running Multi Query Attention Benchmark...

Model                    : MQA
Inference Time (ms)      : 74.571
GPU Memory (MB)          : 0
Tokens/sec               : 13731.929
Parameters               : 590976
KV Cache (MB)            : 0.500


**Comparison on Inference Time**

In [38]:
# ============================================================
# Compare Standard vs MQA
# ============================================================

print("\n")

print("="*90)
print("STANDARD vs MQA")
print("="*90)

print(f"{'Metric':<25}{'Standard':<18}{'MQA':<18}")

print("-"*90)

metrics = [
    "Inference Time (ms)",
    "GPU Memory (MB)",
    "Tokens/sec",
    "Parameters",
    "KV Cache (MB)"
]

for metric in metrics:

    s = result_standard[metric]
    m = result_mqa[metric]

    if isinstance(s,float):
        s = round(s,2)

    if isinstance(m,float):
        m = round(m,2)

    print(f"{metric:<25}{str(s):<18}{str(m):<18}")

print("="*90)



STANDARD vs MQA
Metric                   Standard          MQA               
------------------------------------------------------------------------------------------
Inference Time (ms)      117.38            74.57             
GPU Memory (MB)          0                 0                 
Tokens/sec               8724.17           13731.93          
Parameters               1050624           590976            
KV Cache (MB)            4.0               0.5               
